In [1]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def analyze_and_visualize_solution(sol, data, user_xy, cand, filename="drone_deployment.pdf"):
    status = sol.get("status_str", "")
    if status not in ("OPTIMAL", "TIME_LIMIT"):
        print(f"No feasible solution to analyze (status = {status}).")
        return

    print("\nDETAILED LINK PERFORMANCE")

    t_air_val = sol.get("t_air", 0)
    tech_str = "THz" if t_air_val == 1 else "RF"

    a2g_rf_color  = "mediumseagreen"
    a2a_rf_color  = "darkgreen"
    a2g_thz_color = "royalblue"
    a2a_thz_color = "midnightblue"
    a2g_color = a2g_thz_color if t_air_val == 1 else a2g_rf_color

    assign_raw = sol.get("assignment", {})
    assign = {}
    for u, val in assign_raw.items():
        drones = list(val) if isinstance(val, (list, tuple, set)) else [val]
        if drones:
            assign[u] = drones

    # Print A2G pLoS, SNR, Rate 
    if not assign:
        print("No users are assigned to any drone.")
    else:
        print("\nA2G LINK PERFORMANCE")
        for u in sorted(assign.keys()):
            drones = assign[u]
            total_rate = 0.0
            print(f"User {u:2} is served by {len(drones)} drone(s):")
            for d in drones:
                link_snr  = data["SNR_THz"][(d, u)] if t_air_val == 1 else data["SNR_RF"][(d, u)]
                link_plos = data["pLos_du"][(d, u)]
                link_rate = sol["r_du"][(d, u)]
                total_rate += link_rate

                print(
                    f"  -> Drone {d:3} | Tech: {tech_str} | "
                    f"pLoS: {link_plos:6.4f} | "
                    f"SNR: {link_snr:7.2f} dB | "
                    f"Rate: {link_rate:9.3f} Mbps"
                )

            print(f"     ==> Total rate for user {u:2}: {total_rate:9.3f} Mbps\n")
    
    #Print A2A pLoS, SNR, Rate
    print("\nSELECTED A2A LINK PERFORMANCE")
    rf_links  = sol.get("a2a_rf_links", [])
    thz_links = sol.get("a2a_thz_links", [])

    if not rf_links and not thz_links:
        print("No A2A links are selected.")
    else:
        for (d1, d2) in rf_links:
            print(
                f"  RF  A2A link {d1:3} -> {d2:3} | "
                f"pLoS: {data['pLos_ddp'][(d1,d2)]:6.4f} | "
                f"SNR: {data['SNR_RF_A2A'][(d1,d2)]:7.2f} dB | "
                f"Rate: {data['R_RF_A2A'][(d1,d2)]:9.3f} Mbps"
            )

        for (d1, d2) in thz_links:
            print(
                f"  THz A2A link {d1:3} -> {d2:3} | "
                f"pLoS: {data['pLos_ddp'][(d1,d2)]:6.4f} | "
                f"SNR: {data['SNR_THz_A2A'][(d1,d2)]:7.2f} dB | "
                f"Rate: {data['R_THz_A2A'][(d1,d2)]:9.3f} Mbps"
            )
    # Print average SNR and pLoS for selected links
    print("\nAVERAGE LINK STATISTICS")

    # A2G averages
    a2g_plos_vals = []
    a2g_snr_vals = []

    for u in sorted(assign.keys()):
        drones = assign[u]
        for d in drones:
            a2g_plos_vals.append(data["pLos_du"][(d, u)])
            if t_air_val == 1:
                a2g_snr_vals.append(data["SNR_THz"][(d, u)])
            else:
                a2g_snr_vals.append(data["SNR_RF"][(d, u)])

    if a2g_plos_vals:
        avg_a2g_plos = sum(a2g_plos_vals) / len(a2g_plos_vals)
        avg_a2g_snr  = sum(a2g_snr_vals) / len(a2g_snr_vals)
        print(
            f"A2G ({tech_str}) -> "
            f"Average pLoS: {avg_a2g_plos:.4f} | "
            f"Average SNR: {avg_a2g_snr:.2f} dB"
        )
    else:
        print("A2G -> No selected links.")

    # A2A RF averages
    rf_plos_vals = [data["pLos_ddp"][(d1, d2)] for (d1, d2) in rf_links]
    rf_snr_vals  = [data["SNR_RF_A2A"][(d1, d2)] for (d1, d2) in rf_links]

    if rf_plos_vals:
        avg_rf_plos = sum(rf_plos_vals) / len(rf_plos_vals)
        avg_rf_snr  = sum(rf_snr_vals) / len(rf_snr_vals)
        print(
            f"A2A RF -> "
            f"Average pLoS: {avg_rf_plos:.4f} | "
            f"Average SNR: {avg_rf_snr:.2f} dB"
        )
    else:
        print("A2A RF -> No selected links.")

    # A2A THz averages
    thz_plos_vals = [data["pLos_ddp"][(d1, d2)] for (d1, d2) in thz_links]
    thz_snr_vals  = [data["SNR_THz_A2A"][(d1, d2)] for (d1, d2) in thz_links]

    if thz_plos_vals:
        avg_thz_plos = sum(thz_plos_vals) / len(thz_plos_vals)
        avg_thz_snr  = sum(thz_snr_vals) / len(thz_snr_vals)
        print(
            f"A2A THz -> "
            f"Average pLoS: {avg_thz_plos:.4f} | "
            f"Average SNR: {avg_thz_snr:.2f} dB"
        )
    else:
        print("A2A THz -> No selected links.")
    # Plotting the deployment
    with plt.rc_context({
        "font.family":      "serif",
        "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size":        26,
        "axes.labelsize":   30,
        "axes.labelweight": "bold",
        "xtick.labelsize":  22,
        "ytick.labelsize":  22,
        "legend.fontsize":  30,
    }):
        fig = plt.figure(figsize=(11, 10))
        ax = fig.add_subplot(111, projection="3d")

        # Users
        u_xs = [pos[0] for pos in user_xy]
        u_ys = [pos[1] for pos in user_xy]
        ax.scatter(u_xs, u_ys, 0, c="gray", marker="o", s=30, alpha=0.5)

        # Drones
        masters = sol.get("masters", [])
        slaves  = sol.get("slaves", [])

        for d_idx in masters:
            x, y, h = cand[d_idx]
            ax.scatter(x, y, h, c="red", marker="^", s=120)
            ax.text(x, y, h, f" G{d_idx}", color="red", fontsize=18, fontweight="bold")

        for d_idx in slaves:
            x, y, h = cand[d_idx]
            ax.scatter(x, y, h, c="dimgray", marker="v", s=100)
            ax.text(x, y, h, f" R{d_idx}", color="dimgray", fontsize=18, fontweight="bold")

        # A2G links
        for u_idx, d_list in assign.items():
            ux, uy = user_xy[u_idx]
            if not isinstance(d_list, (list, tuple, set)):
                d_list = [d_list]
            for d_idx in d_list:
                dx, dy, dh = cand[d_idx]
                ax.plot(
                    [ux, dx], [uy, dy], [0, dh],
                    c=a2g_color, linestyle=":", linewidth=1.2, alpha=0.85
                )

        # A2A RF links
        for (d1, d2) in rf_links:
            x1, y1, h1 = cand[d1]
            x2, y2, h2 = cand[d2]
            ax.plot(
                [x1, x2], [y1, y2], [h1, h2],
                c=a2a_rf_color, linestyle="-", linewidth=2.5
            )

        # A2A THz links
        for (d1, d2) in thz_links:
            x1, y1, h1 = cand[d1]
            x2, y2, h2 = cand[d2]
            ax.plot(
                [x1, x2], [y1, y2], [h1, h2],
                c=a2a_thz_color, linestyle="-", linewidth=2.5
            )

        ax.set_xlabel("X Position (m)", labelpad=14)
        ax.set_ylabel("Y Position (m)", labelpad=14)
        ax.set_zlabel("")

        ax.tick_params(axis="both", which="major", labelsize=22, pad=4)
        ax.tick_params(axis="z", which="major", labelsize=22, pad=8)

        for label in ax.get_xticklabels():
            label.set_fontweight("bold")
        for label in ax.get_yticklabels():
            label.set_fontweight("bold")
        for label in ax.get_zticklabels():
            label.set_fontweight("bold")

        legend_handles = [
            Line2D([0], [0], marker="o", color="gray",    linestyle="None", markersize=10, alpha=0.7, label="Users"),
            Line2D([0], [0], marker="^", color="red",     linestyle="None", markersize=13,            label="Gateway"),
            Line2D([0], [0], marker="v", color="dimgray", linestyle="None", markersize=13,            label="Relay"),
            Line2D([0], [0], color=a2g_rf_color,  linestyle=":", linewidth=4.5, label="A2G RF"),
            Line2D([0], [0], color=a2g_thz_color, linestyle=":", linewidth=4.5, label="A2G THz"),
            Line2D([0], [0], color=a2a_rf_color,  linestyle="-", linewidth=4.5, label="A2A RF"),
            Line2D([0], [0], color=a2a_thz_color, linestyle="-", linewidth=4.5, label="A2A THz"),
        ]

        leg = ax.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.04),
            ncol=3,
            frameon=True,
            framealpha=0.95,
            borderpad=0.3,
            labelspacing=0.2,
            handlelength=1.5,
            columnspacing=0.5,
            handletextpad=0.4,
        )

        for text in leg.get_texts():
            text.set_fontweight("bold")

        fig.subplots_adjust(left=0.02, right=0.84, bottom=0.24, top=0.98)

        fig.text(
            0.80, 0.62, "Altitude (m)",
            rotation=90,
            fontsize=26,
            fontweight="bold",
            va="center",
            ha="center",
            family="serif"
        )

        fig.savefig(filename, format="pdf", dpi=600)
        plt.show()

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import random
import math

random.seed(0)

def p_los_itu1410(
    r_km: float,          # horizontal distance r from transmitter to obstacle (km)
    h_tx: float,          # transmitter height (m)
    h_rx: float,          # receiver height (m)
    alpha: float,         # α land area fraction covered by buildings
    beta: float,          # β building density (buildings / km^2)
    gamma: float,         # γ Rayleigh height parameter (m) - the mode
):
    if r_km <= 0:
        return 1 #if the horizontal distance is zero, assume LoS
    # expected number of buildings for r km
    br = int(math.floor(r_km * math.sqrt(alpha * beta)))
    if br <= 0:
        return 1 #if there are no buildings, assume LoS

    sum_weighted = 0.0
    prob_up_to_m = 1.0  # to update with each m
    log_prob = 0.0      # error if not initialized
    for m in range(br):
        di = (m + 0.5) * (r_km / br)
        # height of ray at distance di
        h = h_tx - ((h_tx - h_rx) / r_km) * di

        #CDF of Rayleigh at h
        # probability that a building is smaller than hight hi
        cdf = 1.0 - math.exp(-(h * h) / (2*gamma * gamma))
        cdf = min(max(cdf, 1e-12), 1.0 - 1e-12) #to avoid log(0)

        log_prob += math.log(cdf)
        prob_up_to_m = math.exp(log_prob)
        # 2*m+1 is the weight depending on the distance from the transmitter
        sum_weighted += (2 * m + 1) * prob_up_to_m

    p = sum_weighted / (br * br)
    #just in case
    if p <= 0.0:
        p = 0.0
    elif p > 1.0:
        p = 1.0
    return p

def noise_power_dbm(Bw_hz: float, NF_db: float) -> float:
    #Bw in Hz, noise figure in dB
    pn_db = -174.0 + 10.0 * math.log10(Bw_hz) + NF_db
    return pn_db

def path_loss_rf_db(d_tr: float, f_hz: float, eta_db: float) -> float:
    c = 3e8
    pl_db = 20 * math.log10((4.0 * math.pi * f_hz* d_tr) / c)
    pl_db += eta_db
    return pl_db



import warnings
import numpy as np
from scipy.io import loadmat
from functools import lru_cache

@lru_cache(maxsize=None)
def load_thz_coeffs(coeff_file: str):
    S = loadmat(coeff_file, squeeze_me=True, struct_as_record=False)
    if "coeffs" not in S:
        raise ValueError(f'"{coeff_file}" does not contain a struct named "coeffs".')
    return S["coeffs"]

def _polyval(coeffs, x: float) -> float:
    return float(np.polyval(np.asarray(coeffs, dtype=float).ravel(), x))

def trans_to_pl_local(trans: float, d_km: float, f_thz: float) -> float:
    f_hz = f_thz * 1e12
    d_m = d_km * 1000.0
    c = 299792458.0

    if trans < 0:
        raise ValueError("Transmittance became negative, which is not physical.")

    trans = max(float(trans), np.finfo(float).tiny)

    abs_coeff = (-math.log(trans)) / d_m
    abs_loss_db = abs_coeff * d_m * 10.0 * math.log10(math.e)

    pl_spread_db = (
        20.0 * math.log10(d_m)
        + 20.0 * math.log10(f_hz / 1e6)
        + 20.0 * math.log10(4.0 * math.pi * 1e6 / c)
    )

    return pl_spread_db + abs_loss_db

def g2uav_estimated_pathloss_py(
    model_type: str,
    theta_deg: float,
    d_km: float,
    f_thz: float,
    coeff_file: str = "G2UAV_model_coefficients.mat",
) -> float:
    if model_type not in ("agnostic", "adaptive"):
        raise ValueError("model_type must be 'agnostic' or 'adaptive'")
    if not (0.0 <= theta_deg <= 90.0):
        raise ValueError("theta_deg must be in [0, 90]")
    if d_km <= 0:
        raise ValueError("d_km must be positive")
    if f_thz <= 0:
        raise ValueError("f_thz must be positive")

    if d_km < 0.001 or d_km > 2.5:
        warnings.warn(f"G2UAV: d_km={d_km:.6f} is outside intended range [0.001, 2.5] km.")
    if f_thz < 0.79 or f_thz > 0.83:
        warnings.warn(f"G2UAV: f_THz={f_thz:.6f} is outside intended range [0.79, 0.83] THz.")

    coeffs = load_thz_coeffs(coeff_file)

    if model_type == "adaptive":
        theta_grid = np.asarray(coeffs.theta_adaptive.theta_vec, dtype=float).ravel()
        idx_theta = int(np.argmin(np.abs(theta_grid - theta_deg)))

        lam_all = getattr(coeffs.theta_adaptive, "lambda")
        lam = np.asarray(lam_all[idx_theta], dtype=float).ravel()

        b_est = float(np.polyval(lam, f_thz))
        tau_est = math.exp(b_est * d_km)

    else:  # agnostic
        lambda_h = np.asarray(coeffs.theta_agnostic.lambda_h, dtype=float).ravel()
        lambda_v = np.asarray(coeffs.theta_agnostic.lambda_v, dtype=float).ravel()

        kh = float(np.polyval(lambda_h, f_thz))
        kv = float(np.polyval(lambda_v, f_thz))

        # d_h = d * sin(theta), d_v = d * cos(theta)
        d_h = d_km * math.sin(math.radians(theta_deg))
        d_v = d_km * math.cos(math.radians(theta_deg))

        tau_est = math.exp(kh * d_h + kv * d_v)

    tau_est = max(tau_est, np.finfo(float).tiny)
    return trans_to_pl_local(tau_est, d_km, f_thz)

# UAV2UAV THz path loss
# l_km = minimum altitude between Tx and Rx
def uav2uav_estimated_pathloss_py(
    model_type: str,
    l_km: float,
    theta_deg: float,
    d_km: float,
    f_thz: float,
    coeff_file: str = "UAV2UAV_model_coefficients.mat",
) -> float:
    if model_type not in ("agnostic", "adaptive"):
        raise ValueError("model_type must be 'agnostic' or 'adaptive'")
    if l_km <= 0:
        raise ValueError("l_km must be positive")
    if not (0.0 <= theta_deg <= 90.0):
        raise ValueError("theta_deg must be in [0, 90]")
    if d_km <= 0:
        raise ValueError("d_km must be positive")
    if f_thz <= 0:
        raise ValueError("f_thz must be positive")

    if l_km < 0.03 or l_km > 2.0:
        warnings.warn(f"UAV2UAV: l_km={l_km:.6f} is outside intended range [0.03, 2.0] km.")
    if d_km < 0.01 or d_km > 2.43:
        warnings.warn(f"UAV2UAV: d_km={d_km:.6f} is outside intended range [0.01, 2.43] km.")
    if f_thz < 0.79 or f_thz > 0.83:
        warnings.warn(f"UAV2UAV: f_THz={f_thz:.6f} is outside intended range [0.79, 0.83] THz.")

    coeffs = load_thz_coeffs(coeff_file)

    # theta=0 -> vertical, theta=90 -> horizontal
    d_h = d_km * math.sin(math.radians(theta_deg))
    d_v = d_km * math.cos(math.radians(theta_deg))

    if model_type == "adaptive":
        theta_grid = np.asarray(coeffs.theta_adaptive.theta_vec, dtype=float).ravel()
        idx_theta = int(np.argmin(np.abs(theta_grid - theta_deg)))

        lam_all = getattr(coeffs.theta_adaptive, "lambda")
        lam = np.asarray(lam_all[idx_theta], dtype=float).ravel()

        b2_all = np.asarray(coeffs.theta_adaptive.b2, dtype=float).ravel()
        b2_est = float(b2_all[idx_theta])

        a2_est = float(np.polyval(lam, f_thz))
        slope_est = a2_est * math.exp(b2_est * l_km)
        tau_est = math.exp(slope_est * d_km)

    else:  # agnostic
        lambda_h = np.asarray(coeffs.theta_agnostic.lambda_h, dtype=float).ravel()
        lambda_v = np.asarray(coeffs.theta_agnostic.lambda_v, dtype=float).ravel()
        b2_h = float(coeffs.theta_agnostic.b2_h)
        b2_v = float(coeffs.theta_agnostic.b2_v)

        Lambda_h = float(np.polyval(lambda_h, f_thz))
        Lambda_v = float(np.polyval(lambda_v, f_thz))

        tau_est = math.exp(
            Lambda_h * math.exp(b2_h * l_km) * d_h
            + Lambda_v * math.exp(b2_v * l_km) * d_v
        )

    tau_est = float(np.clip(tau_est, np.finfo(float).tiny, 1.0))
    return trans_to_pl_local(tau_est, d_km, f_thz)

def pathloss_thz_a2g_geom_db(
    d_tr: float,
    h_tx: float,
    h_rx: float,
    f_thz: float,
    coeff_file: str = "G2UAV_model_coefficients.mat",
    model_type: str = "agnostic",
) -> float:
    if d_tr <= 0:
        return 0.0

    vert_m = abs(h_tx - h_rx)
    horiz_m = math.sqrt(max(d_tr**2 - vert_m**2, 0.0))

    # zenith angle: 0° vertical, 90° horizontal
    theta_deg = math.degrees(math.atan2(horiz_m, vert_m)) if vert_m > 0 else 90.0
    d_km = d_tr / 1000.0

    return g2uav_estimated_pathloss_py(
        model_type=model_type,
        theta_deg=theta_deg,
        d_km=d_km,
        f_thz=f_thz,
        coeff_file=coeff_file,
    )


def pathloss_thz_a2a_geom_db(
    d_tr: float,
    h_tx: float,
    h_rx: float,
    f_thz: float,
    coeff_file: str = "UAV2UAV_model_coefficients.mat",
    model_type: str = "agnostic",
) -> float:
    if d_tr <= 0:
        return 0.0

    vert_m = abs(h_tx - h_rx)
    horiz_m = math.sqrt(max(d_tr**2 - vert_m**2, 0.0))

    # zenith angle: 0° vertical, 90° horizontal
    theta_deg = math.degrees(math.atan2(horiz_m, vert_m)) if vert_m > 0 else 90.0
    l_km = min(h_tx, h_rx) / 1000.0
    d_km = d_tr / 1000.0

    return uav2uav_estimated_pathloss_py(
        model_type=model_type,
        l_km=l_km,
        theta_deg=theta_deg,
        d_km=d_km,
        f_thz=f_thz,
        coeff_file=coeff_file,
    )

def SNR_linear(
    p_tx_dbm: float,
    g_tx_db: float,
    g_rx_db: float,
    pl_db: float,
    pn_db: float,
) -> float:
    snr_db = p_tx_dbm + g_tx_db + g_rx_db - pl_db - pn_db
    return 10 ** (snr_db / 10)

def SNR_expected_rf(
    r_km: float,
    d_tr: float,
    h_tx: float,
    h_rx: float,
    alpha: float,
    beta: float,
    gamma: float,
    f_rf_hz: float,
    f_thz_hz: float,
    Bw_rf_hz: float,
    Bw_thz_hz: float,
    Pt_dbm: float,
    gt_dbi: float,
    gr_dbi: float,
    NF_rf_db: float,
    NF_thz_db: float,
    eta_los_db: float,
    eta_nlos_db: float,
    is_a2g: bool,
    thz_model_type: str = "agnostic",
) -> dict:
    #RF
    p_los = p_los_itu1410(r_km, h_tx, h_rx, alpha, beta, gamma)
    Pn_db_rf = noise_power_dbm(Bw_rf_hz, NF_rf_db)

    pl_los_db_rf = path_loss_rf_db(d_tr, f_rf_hz, eta_los_db)
    pl_nlos_db_rf = path_loss_rf_db(d_tr, f_rf_hz, eta_nlos_db)

    snr_los_rf = SNR_linear(Pt_dbm, gt_dbi, gr_dbi, pl_los_db_rf, Pn_db_rf)
    snr_nlos_rf = SNR_linear(Pt_dbm, gt_dbi, gr_dbi, pl_nlos_db_rf, Pn_db_rf)

    snr_rf_exp = p_los * snr_los_rf + (1.0 - p_los) * snr_nlos_rf
    snr_rf_exp_db = 10.0 * math.log10(snr_rf_exp) if snr_rf_exp > 0 else -100.0
    #THz
    Pn_db_thz = noise_power_dbm(Bw_thz_hz, NF_thz_db)

    Pt_thz_dbm = 20.0
    gt_thz_dbi = 30.0
    gr_thz_dbi = 25.0

    if is_a2g:
        pl_db_thz = pathloss_thz_a2g_geom_db(
            d_tr=d_tr,
            h_tx=h_tx,
            h_rx=h_rx,
            f_thz=f_thz_hz,
            coeff_file="G2UAV_model_coefficients.mat",
            model_type=thz_model_type,
        )
    else:
        pl_db_thz = pathloss_thz_a2a_geom_db(
            d_tr=d_tr,
            h_tx=h_tx,
            h_rx=h_rx,
            f_thz=f_thz_hz,
            coeff_file="UAV2UAV_model_coefficients.mat",
            model_type=thz_model_type,
        )

    snr_thz = SNR_linear(Pt_thz_dbm, gt_thz_dbi, gr_thz_dbi, pl_db_thz, Pn_db_thz)
    snr_thz_db = 10.0 * math.log10(snr_thz) if snr_thz > 0 else -100.0

    return {
        "p_los": p_los,
        "snr_los_rf": snr_los_rf,
        "snr_nlos_rf": snr_nlos_rf,
        "snr_rf_exp_lin": snr_rf_exp,
        "snr_thz_lin": snr_thz,
        "snr_rf_exp_db": snr_rf_exp_db,
        "snr_thz_db": snr_thz_db,
    }

def data_rate_rf(pLos_du: float, snr_los_rf: float, snr_nlos_rf:float, Bw_hz: float) -> float:
    rate_bps_rf = pLos_du * Bw_hz * math.log2(1 + snr_los_rf) + (1 - pLos_du) * Bw_hz * math.log2(1 + snr_nlos_rf)
    return rate_bps_rf

def data_rate_thz(pLos_du: float, snr_thz_lin: float, Bw_hz: float) -> float:
    rate_bps_thz = pLos_du * Bw_hz * math.log2(1 + snr_thz_lin)
    return rate_bps_thz

num_users = 50
U = list(range(num_users))
user_xy = [(random.uniform(0, 1000), random.uniform(0, 1000)) for _ in U] #different user distributions are used across scenarios. Uniform and dclustering.
#below is for creating clustered users
'''
num_clusters =5              # how many hotspots
cluster_std = 90              # spread of each cluster (smaller means tighter clusters)
cluster_centers = [           # Random cluster centers inside the area
    (random.uniform(x_min + 100, x_max - 100),
     random.uniform(y_min + 100, y_max - 100))
    for _ in range(num_clusters)
]
user_xy = []
for _ in U:
    cx, cy = random.choice(cluster_centers)
    # Gaussian around the chosen center
    x = random.gauss(cx, cluster_std)
    y = random.gauss(cy, cluster_std)

    # keep inside bounds
    x = max(x_min, min(x_max, x))
    y = max(y_min, min(y_max, y))

    user_xy.append((x, y))
'''

altitudes = [90,100]

x_min, x_max, dx = 0, 1000, 100  #dx and dy are the step sizes. When they are too small, the time it takes for gurobi to find the optimal scenario increases very significantly.
y_min, y_max, dy = 0, 1000, 100 

xy_points = [(x, y)
            for x in range(x_min, x_max + 1, dx)
            for y in range(y_min, y_max + 1, dy)]
# Candidate drones, all combinations of (x,y) points and altitudes
cand = [(x, y, h) for (x, y) in xy_points for h in altitudes]

D = list(range(len(cand)))

print("Number of users:", len(U))
print("Number of XY points:", len(xy_points))
print("XY points:", xy_points)
print("Altitude levels:", altitudes)
print("Number of candidates:", len(D))
print(cand)

# A2G
SNR_RF = {}
SNR_THz = {}
R_RF = {}
R_THz = {}
pLos_du = {}
#parameters
Pt_dbm = 20 #Tx power
gt_dbi = 5.0 #Tx antenna gain
gr_dbi = 0.0 #Rx antenna gain
#RF
f_rf_hz   = 2.0e9
Bw_rf_hz  = 20e6 
NF_rf_db  = 7.0
eta_los_db  = 1.6
eta_nlos_db = 23.0      #(ηLoS,ηNLoS) pairs are:
                        #(0.1 dB, 21 dB) -suburban
                        #(1 dB, 20 dB) - urban,
                        # (1.6 dB, 23 dB) - dense urban,
                        # (2.3 dB,34 dB) - highrise urban
#Thz
f_thz_hz  = 0.8    # 0.79-0.91 and 0.93-0.94 THz bands are allowed (band1/band2)
Bw_thz_hz = 0.5e9
NF_thz_db = 10.0
#environment params for Los probability
alpha = 0.5  #land area fraction covered by buildings
beta = 550 #building density (buildings/km^2)
gamma = 35 # Rayleigh height parameter (m)
#                 α      β    γ
#Suburban        0.11   750   8
#Urban           0.3    650   20
#Dense Urban     0.5    550   35
#High-rise Urban 0.7    450   50

#A2G
for d in D:
    xd, yd, hd = cand[d]
    for u in U:
        xu, yu = user_xy[u]
        dist = math.sqrt((xd - xu)**2 + (yd - yu)**2 + hd**2)
        dist_horiz_m = math.sqrt((xd - xu)**2 + (yd - yu)**2) #horizontal distance for LoS probability
        r_km = dist_horiz_m / 1000.0 #to km
        pLos_du[(d, u)] = p_los_itu1410(
            r_km=r_km,
            h_tx=hd,
            h_rx=0,
            alpha=alpha,
            beta=beta,
            gamma=gamma
        )
        SNR_results = SNR_expected_rf(
            r_km=r_km,
            d_tr=dist,
            h_tx=hd,
            h_rx=0,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            f_rf_hz=f_rf_hz,
            f_thz_hz=f_thz_hz,
            Bw_rf_hz=Bw_rf_hz,
            Bw_thz_hz=Bw_thz_hz,
            Pt_dbm=Pt_dbm,
            gt_dbi=gt_dbi,
            gr_dbi=gr_dbi,
            NF_rf_db=NF_rf_db,
            NF_thz_db=NF_thz_db,
            eta_los_db=eta_los_db,
            eta_nlos_db=eta_nlos_db,
            is_a2g=True,
            thz_model_type="agnostic",   # or "adaptive"
        )
        snr_rf_exp_db = SNR_results["snr_rf_exp_db"]
        snr_thz_db    = SNR_results["snr_thz_db"]
        snr_rf_exp_lin = SNR_results["snr_rf_exp_lin"]
        rate_bps_rf = data_rate_rf(pLos_du=pLos_du[(d, u)], snr_los_rf=SNR_results["snr_los_rf"], snr_nlos_rf=SNR_results["snr_nlos_rf"], Bw_hz=Bw_rf_hz)
        rate_bps_thz = data_rate_thz(pLos_du=pLos_du[(d, u)], snr_thz_lin=SNR_results["snr_thz_lin"], Bw_hz=Bw_thz_hz)
        SNR_RF[(d, u)] = snr_rf_exp_db
        SNR_THz[(d, u)] = snr_thz_db
        R_RF[(d, u)] = rate_bps_rf/1e6
        R_THz[(d, u)] = rate_bps_thz/1e6

# A2A
SNR_RF_A2A = {}
SNR_THz_A2A = {}
R_RF_A2A = {}
R_THz_A2A = {}
pLos_ddp = {}
dist_A2A = {}
for d in D:
    xd, yd, hd = cand[d]
    for dp in D:
        if dp == d:
            continue
        x2, y2, h2 = cand[dp]
        dist = math.sqrt((xd - x2)**2 + (yd - y2)**2 + (hd - h2)**2) #euclidean distance
        dist_A2A[(d, dp)] = dist
        dist_horiz_m = math.sqrt((xd - x2)**2 + (yd - y2)**2) #horizontal distance for LoS probability
        r_km = dist_horiz_m / 1000.0 #to km
        pLos_ddp[(d, dp)] = p_los_itu1410(
            r_km=r_km,
            h_tx=hd,
            h_rx=h2,
            alpha=alpha,
            beta=beta,
            gamma=gamma
        )
        SNR_results_a2a = SNR_expected_rf(
            r_km=r_km,
            d_tr=dist,
            h_tx=hd,
            h_rx=h2,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            f_rf_hz=f_rf_hz,
            f_thz_hz=f_thz_hz,
            Bw_rf_hz=Bw_rf_hz,
            Bw_thz_hz=Bw_thz_hz,
            Pt_dbm=Pt_dbm,
            gt_dbi=gt_dbi,
            gr_dbi=gr_dbi,
            NF_rf_db=NF_rf_db,
            NF_thz_db=NF_thz_db,
            eta_los_db=eta_los_db,
            eta_nlos_db=eta_nlos_db,
            is_a2g=False,
            thz_model_type="agnostic",   # or "adaptive"
        )
        snr_rf_exp_db_a2a = SNR_results_a2a["snr_rf_exp_db"]
        snr_thz_db_a2a    = SNR_results_a2a["snr_thz_db"]
        rate_bps_rf_a2a = data_rate_rf(pLos_du=pLos_ddp[(d, dp)], snr_los_rf=SNR_results_a2a["snr_los_rf"], snr_nlos_rf=SNR_results_a2a["snr_nlos_rf"], Bw_hz=Bw_rf_hz)
        rate_bps_thz_a2a = data_rate_thz(pLos_du=pLos_ddp[(d, dp)], snr_thz_lin=SNR_results_a2a["snr_thz_lin"], Bw_hz=Bw_thz_hz)
        SNR_RF_A2A[(d, dp)] = snr_rf_exp_db_a2a
        SNR_THz_A2A[(d, dp)] = snr_thz_db_a2a
        R_RF_A2A[(d, dp)] = rate_bps_rf_a2a/1e6
        R_THz_A2A[(d, dp)] = rate_bps_thz_a2a/1e6

#values are called from this dictionary
data = dict(
    D=D,
    U=U,
    Cost_master=40.0,
    Cost_slave=6.0,

    Rmin=5,  #Mbps
    SNR_A2G_RF_min=5.0,
    SNR_A2G_THz_min=5.0,
    SNR_A2A_RF_min=5.0,
    SNR_A2A_THz_min=5.0,

    pLoS_THz_min=0.8,

    SNR_RF=SNR_RF,
    SNR_THz=SNR_THz,
    R_RF=R_RF,
    R_THz=R_THz,
    pLos_du=pLos_du,
    SNR_RF_A2A=SNR_RF_A2A,
    SNR_THz_A2A=SNR_THz_A2A,
    R_RF_A2A=R_RF_A2A,
    R_THz_A2A=R_THz_A2A,
    pLos_ddp=pLos_ddp,
    dist_A2A=dist_A2A
)
def solve_drone_deployment_gurobi(data, time_limit_s=math.inf, mip_gap=0.1, verbose=True):
    #read the dictionary "data". Build the gurobi model. Solve it. Return the solution.
    D = data["D"]
    U = data["U"]

    Cost_master = float(data["Cost_master"])
    Cost_slave  = float(data["Cost_slave"])

    Rmin = float(data["Rmin"])
    SNR_A2G_RF_min  = float(data["SNR_A2G_RF_min"])
    SNR_A2G_THz_min = float(data["SNR_A2G_THz_min"])
    SNR_A2A_RF_min  = float(data["SNR_A2A_RF_min"])
    SNR_A2A_THz_min = float(data["SNR_A2A_THz_min"])

    pLoS_THz_min = float(data["pLoS_THz_min"])

    SNR_RF   = data["SNR_RF"]
    SNR_THz  = data["SNR_THz"]
    R_RF     = data["R_RF"]
    R_THz    = data["R_THz"]
    pLos_du  = data["pLos_du"]

    SNR_RF_A2A  = data["SNR_RF_A2A"]
    SNR_THz_A2A = data["SNR_THz_A2A"]
    R_RF_A2A    = data["R_RF_A2A"]
    R_THz_A2A   = data["R_THz_A2A"]
    pLos_ddp    = data["pLos_ddp"]
    dist_A2A    = data["dist_A2A"]

    # Big-M must be large enough to relax the constraint when g=0:
    min_snr_a2a = min(
        min(SNR_RF_A2A[(d, dp)], SNR_THz_A2A[(d, dp)])
        for d in D for dp in D if dp != d
    )

    M_snr_a2a = (max(SNR_A2A_RF_min, SNR_A2A_THz_min) - min_snr_a2a) + 1.0
    M_cap = (len(U) * 100) + 1

    m = gp.Model("drone_deployment_discrete")
    m.Params.TimeLimit = time_limit_s
    m.Params.MIPGap = mip_gap #Stop when the solution is within mip_gap of optimal.
    # Decision variables
    # addVars(): Create one variable for each element of D.
    z = m.addVars(D, vtype=GRB.BINARY, name="z")        # drone deployed or not
    w = m.addVars(D, vtype=GRB.BINARY, name="w")       # deployed drone master or not
    x = m.addVars(D, U, vtype=GRB.BINARY, name="x")    # user u served by drone d or not

    t_air   = m.addVar(vtype=GRB.BINARY, name="t_air")    # A2G tech selector  0-RF, 1-THz
    t_space = m.addVar(vtype=GRB.BINARY, name="t_space") # S2A tech selector 0-RF, 1-THz

    a = m.addVars(D, U, vtype=GRB.BINARY, name="a")  # a = x AND t_air=0 , for SNR and data rate constraints
    b = m.addVars(D, U, vtype=GRB.BINARY, name="b")  # b = x AND t_air=1

    #actual rate from drone d to user u (in Mbps)
    r = m.addVars(D, U, vtype=GRB.CONTINUOUS, lb=0.0, name="r")

    # share of RF/THz "bandwidth/time" that drone d gives to user u
    theta_RF  = m.addVars(D, U, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="theta_RF")
    theta_THz = m.addVars(D, U, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="theta_THz")

    gRF  = m.addVars(D, D, vtype=GRB.BINARY, name="gRF")   # A2A RF link 1 or 0
    gTHz = m.addVars(D, D, vtype=GRB.BINARY, name="gTHz")   # A2A THz link 1 or 0

    # Actual A2A forwarded flow from drone d to drone dp, in Mbps
    fA2A = m.addVars(D, D, vtype=GRB.CONTINUOUS, lb=0.0, name="fA2A")

    # A2A bandwidth/time share used on each A2A link
    phi_A2A_RF = m.addVars(D, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="phi_A2A_RF")
    phi_A2A_THz = m.addVars(D, D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="phi_A2A_THz")

    # Slave indicator s_d = z_d - w_d (binary because w<=z)
    s = m.addVars(D, vtype=GRB.BINARY, name="s_slave")

    m.setObjective(
        gp.quicksum(w[d] * Cost_master + (z[d] - w[d]) * Cost_slave for d in D),
        GRB.MINIMIZE
    )
    # Constraints
    # Deployment constraints:
    # At least one master for deployment: sum_d z_d w_d >= 1  equivalently sum_d w_d >= 1 since w_d <= z_d
    m.addConstr(
        gp.quicksum(w[d] for d in D) >= 1,
        name="at_least_one_master"
    )
    m.addConstrs((w[d] <= z[d] for d in D), name="master_only_if_deployed")
    m.addConstrs((s[d] == z[d] - w[d] for d in D), name="slave_indicator")
    # Each user served by at least one drone and only if deployed.
    m.addConstrs((gp.quicksum(x[d, u] for d in D) >= 1 for u in U), name="at_least_one_serving_drone")
    m.addConstrs((x[d, u] <= z[d] for d in D for u in U), name="no_connection_with_non_deployed_drone")
    eps = 1e-3
    for d in D:
        for u in U:
            # a = x AND (1 - t_air)   deployed and RF
            m.addConstr(a[d,u] <= x[d,u])
            m.addConstr(a[d,u] <= 1 - t_air)
            m.addConstr(a[d,u] >= x[d,u] - t_air)
            # b = x * t_air    deployed and THz
            m.addConstr(b[d,u] <= x[d,u])
            m.addConstr(b[d,u] <= t_air)
            m.addConstr(b[d,u] >= x[d,u] + t_air - 1)

            m.addConstr(a[d,u] + b[d,u] == x[d,u], name=f"splitTech_{d}_{u}")

            snr_selected = SNR_RF[d,u]*a[d,u] + SNR_THz[d,u]*b[d,u]
            m.addConstr(snr_selected >= SNR_A2G_RF_min*a[d,u] + SNR_A2G_THz_min*b[d,u])
            # With bandwidth splitting:
            # r_du <= (full-band RF rate)*theta_RF + (full-band THz rate)*theta_THz
            m.addConstr(
                r[d, u] <= R_RF[d, u] * theta_RF[d, u]
                         + R_THz[d, u] * theta_THz[d, u],
                name=f"link_rate_cap_split_{d}_{u}"
            )
            # share only if link is active with that tech
            m.addConstr(theta_RF[d, u]  <= a[d, u], name=f"thetaRF_le_a_{d}_{u}")
            m.addConstr(theta_THz[d, u] <= b[d, u], name=f"thetaTHz_le_b_{d}_{u}")
            # no assignment without traffic
            m.addConstr(
                r[d, u] >= eps * x[d, u],
                name=f"min_rate_if_assigned_{d}_{u}"
            )
    # Common gateway bandwidth sharing:
    # A gateway's total bandwidth is shared between:
    #   1) direct A2G links from gateway to users
    #   2) incoming A2A links from relay/slave drones
    for d in D:
        m.addConstr(
            gp.quicksum(theta_RF[d, u] for u in U)
            + gp.quicksum(phi_A2A_RF[d, dp] for dp in D if dp != d)   # outgoing (slave)
            + gp.quicksum(phi_A2A_RF[dp, d] for dp in D if dp != d)   # incoming (master)
            <= 1.0,
            name=f"joint_RF_budget_{d}"
        )
        m.addConstr(
            gp.quicksum(theta_THz[d, u] for u in U)
            + gp.quicksum(phi_A2A_THz[d, dp] for dp in D if dp != d)  # outgoing (slave)
            + gp.quicksum(phi_A2A_THz[dp, d] for dp in D if dp != d)  # incoming (master)
            <= 1.0,
            name=f"joint_THz_budget_{d}"
        )
    for u in U:
        m.addConstr(
            gp.quicksum(r[d, u] for d in D) >= Rmin,
            name=f"RateMin_{u}"
        )
    # A2A Master–Slave constraints
    # No self-links
    for d in D:
        m.addConstr(gRF[d, d] == 0,  name=f"no_self_RF_{d}")
        m.addConstr(gTHz[d, d] == 0, name=f"no_self_THz_{d}")
    # t_space = 0 the n only RF A2A links allowed  -all gTHz must be 0
    # t_space = 1 then only THz A2A links allowed -all gRF  must be 0
    for d in D:
        for dp in D:
            if dp == d:
                continue
            # gRF[d,dp]  can only be 1 when t_space = 0
            m.addConstr(gRF[d, dp]  <= 1 - t_space, name=f"gRF_tech_global_{d}_{dp}")
            # gTHz[d,dp] can only be 1 when t_space = 1
            m.addConstr(gTHz[d, dp] <= t_space,  name=f"gTHz_tech_global_{d}_{dp}")

    # Only deployed nodes can have A2A links, g links exist only from a SLAVE d to a MASTER d' and both deployed.
    for d in D:
        for dp in D:
            if dp == d:
                continue
            for gvar, tech in [(gRF, "RF"), (gTHz, "THz")]:
                m.addConstr(gvar[d, dp] <= z[d],        name=f"g{tech}_le_z_from_{d}_{dp}")
                m.addConstr(gvar[d, dp] <= (1 - w[d]),  name=f"g{tech}_le_slave_from_{d}_{dp}")
                m.addConstr(gvar[d, dp] <= w[dp],       name=f"g{tech}_le_master_to_{d}_{dp}")
                m.addConstr(gvar[d, dp] <= z[dp],       name=f"g{tech}_le_z_to_{d}_{dp}")
    # Each slave connects to at least one master:
    for d in D:
        m.addConstr(
            gp.quicksum((gRF[d, dp] + gTHz[d, dp]) for dp in D if dp != d) >= s[d],
            name=f"slave_connects_{d}"
        )
    # A2A bandwidth sharing and actual forwarded flow
    # No self-flow and no self-bandwidth share
    for d in D:
        m.addConstr(fA2A[d, d] == 0, name=f"no_self_flow_A2A_{d}")
        m.addConstr(phi_A2A_RF[d, d] == 0, name=f"no_self_phi_RF_A2A_{d}")
        m.addConstr(phi_A2A_THz[d, d] == 0, name=f"no_self_phi_THz_A2A_{d}")
    for d in D:
        for dp in D:
            if dp == d:
                continue
            # A2A flow cannot exceed the shared-capacity of the selected A2A link
            m.addConstr(
                fA2A[d, dp] <= R_RF_A2A[d, dp] * phi_A2A_RF[d, dp]
                            + R_THz_A2A[d, dp] * phi_A2A_THz[d, dp],
                name=f"A2A_flow_capacity_with_sharing_{d}_{dp}"
            )
            # RF share can only be positive if the RF A2A link is selected
            m.addConstr(
                phi_A2A_RF[d, dp] <= gRF[d, dp],
                name=f"phi_A2A_RF_le_gRF_{d}_{dp}"
            )
            # THz share can only be positive if the THz A2A link is selected
            m.addConstr(
                phi_A2A_THz[d, dp] <= gTHz[d, dp],
                name=f"phi_A2A_THz_le_gTHz_{d}_{dp}"
            )
    # Flow conservation for slaves:
    # Traffic served by a slave must be forwarded to master/s
    for d in D:
        slave_access_load = gp.quicksum(r[d, u] for u in U)
        slave_forwarded_load = gp.quicksum(fA2A[d, dp] for dp in D if dp != d)
        # If d is a slave, w[d] = 0, so these become equality.
        # If d is a master, w[d] = 1, so they are relaxed.
        m.addConstr(
            slave_forwarded_load >= slave_access_load - M_cap * w[d],
            name=f"slave_flow_conservation_lower_{d}"
        )
        m.addConstr(
            slave_forwarded_load <= slave_access_load + M_cap * w[d],
            name=f"slave_flow_conservation_upper_{d}"
        )
    # A2G THz LoS gating
    for d in D:
        for u in U:
            m.addConstr(
                pLos_du[(d, u)] >= pLoS_THz_min * b[d, u],
                name=f"A2G_THz_LoS_gate_{d}_{u}"
            )
    # A2A SNR requirement:
    for d in D:
        for dp in D:
            if dp == d:
                continue
            # RF
            m.addConstr(
                SNR_RF_A2A[d, dp] + M_snr_a2a * (1 - gRF[d, dp]) >= SNR_A2A_RF_min,
                name=f"A2A_SNR_RF_{d}_{dp}"
            )
            # THz
            m.addConstr(
                SNR_THz_A2A[d, dp] + M_snr_a2a * (1 - gTHz[d, dp]) >= SNR_A2A_THz_min,
                name=f"A2A_SNR_THz_{d}_{dp}"
            )
    # A2A THz LoS gating:
    for d in D:
        for dp in D:
            if dp == d:
                continue
            m.addConstr(
                pLos_ddp[d, dp] >= pLoS_THz_min * gTHz[d, dp],
                name=f"A2A_THz_LoS_gate_{d}_{dp}"
            )
    # Solve
    m.optimize()
    status = m.Status
    if status == GRB.INFEASIBLE:
        m.computeIIS()
        m.write("model.ilp")
        return {
            "status": status,
            "status_str": "INFEASIBLE",
            "iis_file": "model.ilp"
        }
    if status in [GRB.UNBOUNDED, GRB.INF_OR_UNBD]:
        return {
            "status": status,
            "status_str": "UNBOUNDED/INF_OR_UNBD"
        }
    # If time limit, only read solution if an incumbent exists
    if status == GRB.TIME_LIMIT and m.SolCount == 0:
        return {
            "status": status,
            "status_str": "TIME_LIMIT_NO_SOLUTION"
        }
    sol = {
        "status": status,
        "status_str": {
            GRB.OPTIMAL: "OPTIMAL",
            GRB.TIME_LIMIT: "TIME_LIMIT"
        }.get(status, str(status)),
        "obj": m.ObjVal
    }
    sol["t_air"] = int(round(t_air.X))
    sol["t_space"] = int(round(t_space.X))
    sol["deployed"] = [d for d in D if z[d].X > 0.5]
    sol["masters"]  = [d for d in D if w[d].X > 0.5]
    sol["slaves"]   = [d for d in D if s[d].X > 0.5]
    user_rates = {}
    for u in U:
        user_rates[u] = sum(r[d, u].X for d in D)
    sol["user_rates"] = user_rates

    drone_rates = {}
    for d in D:
        drone_rates[d] = sum(r[d, u].X for u in U)
    sol["drone_rates"] = drone_rates
    # Assignments: for each user, list of serving drones
    assign = {}
    for u in U:
        serving_drones = []
        for d in D:
            if x[d, u].X > 0.5:
                serving_drones.append(d)
        if serving_drones:
            assign[u] = serving_drones
    sol["assignment"] = assign

    # A2A links
    links_rf = []
    links_thz = []
    for d in D:
        for dp in D:
            if d == dp:
                continue
            if gRF[d, dp].X > 0.5:
                links_rf.append((d, dp))
            if gTHz[d, dp].X > 0.5:
                links_thz.append((d, dp))
    sol["a2a_rf_links"] = links_rf
    sol["a2a_thz_links"] = links_thz
    sol["r_du"] = {(d,u): r[d,u].X for d in D for u in U}
    sol["a2a_flow"] = {
    (d, dp): fA2A[d, dp].X
    for d in D for dp in D
    if d != dp and fA2A[d, dp].X > 1e-6
}
    sol["a2a_phi_rf"] = {
        (d, dp): phi_A2A_RF[d, dp].X
        for d in D for dp in D
        if d != dp and phi_A2A_RF[d, dp].X > 1e-6
    }
    sol["a2a_phi_thz"] = {
        (d, dp): phi_A2A_THz[d, dp].X
        for d in D for dp in D
        if d != dp and phi_A2A_THz[d, dp].X > 1e-6
    }
    return sol

if __name__ == "__main__":
    sol = solve_drone_deployment_gurobi(data, time_limit_s=math.inf, mip_gap=0.1, verbose=True)
    print("\n SOLUTION SUMMARY")
    print("Status:", sol.get("status_str"), "Obj:", sol.get("obj"))
    print("t_air (0 RF, 1 THz):", sol.get("t_air"))
    print("Deployed:", sol.get("deployed"))
    print("Masters :", sol.get("masters"))
    print("Slaves  :", sol.get("slaves"))

    print("\nA2A LINKS ")
    print("RF links :", sol.get("a2a_rf_links"))
    print("THz links:", sol.get("a2a_thz_links"))

    print("\nUSER ASSIGNMENTS (first 20)")
    assign = sol.get("assignment", {})
    for i, (u, d) in enumerate(assign.items()):
        if i >= 20:
            break
        print(f"User {u} -> Drone {d}")

    print(sol["status_str"], "obj=", sol.get("obj"))
    print("t_air=", sol.get("t_air"), "masters=", sol.get("masters"), "slaves=", sol.get("slaves"))
    print("\nA2A FORWARDED FLOWS")
    for link, flow in sol.get("a2a_flow", {}).items():
        print(f"{link}: {flow:.3f} Mbps")

    print("\nA2A RF SHARES")
    for link, share in sol.get("a2a_phi_rf", {}).items():
        print(f"{link}: {share:.3f}")

    print("\nA2A THz SHARES")
    for link, share in sol.get("a2a_phi_thz", {}).items():
        print(f"{link}: {share:.3f}")


analyze_and_visualize_solution(sol, data, user_xy, cand)